In [1]:
import leafmap
import ipywidgets as widgets
from IPython.display import display
import rasterio
from rasterio.warp import transform_bounds

# 1. Create the User Interface (UI) Elements
file_input = widgets.Text(
    value=r"C:\Users\shash\Downloads\projects\hstool\hybiomass_south_asia\hybiomass_dataset\South_Asia\01931\024.tif",
    placeholder='Enter full path to .tif file',
    description='File Path:',
    layout=widgets.Layout(width='80%')
)

# Band selection widgets
r_band = widgets.IntText(value=45, description='Red Band:', layout=widgets.Layout(width='150px'))
g_band = widgets.IntText(value=29, description='Green Band:', layout=widgets.Layout(width='150px'))
b_band = widgets.IntText(value=15, description='Blue Band:', layout=widgets.Layout(width='150px'))

# Contrast stretching widgets
vmin_input = widgets.IntText(value=110, description='Vmin:', layout=widgets.Layout(width='200px'))
vmax_input = widgets.IntText(value=2000, description='Vmax:', layout=widgets.Layout(width='200px'))

# The action button and output area
render_button = widgets.Button(description="Render Map", button_style='primary')
output_map = widgets.Output()

# 2. Helper function to safely calculate Lat/Lon center from the UTM file
def get_map_center(filepath):
    with rasterio.open(filepath) as src:
        bounds = src.bounds
        crs = src.crs
        # Transform the bounding box to EPSG:4326 (Lat/Lon)
        left, bottom, right, top = transform_bounds(crs, 'EPSG:4326', *bounds)
        lat = (bottom + top) / 2
        lon = (left + right) / 2
    return lat, lon

# 3. The main logic that runs when you click the button
def on_button_clicked(b):
    with output_map:
        # Clear the previous map if it exists
        output_map.clear_output() 
        
        filepath = file_input.value
        try:
            # Automatically calculate the exact center in degrees
            lat, lon = get_map_center(filepath)
            
            # Initialize map
            m = leafmap.Map(center=[lat, lon], zoom=14)
            
            # Add the raster using the values from the text boxes
            m.add_raster(
                filepath, 
                bands=[r_band.value, g_band.value, b_band.value], 
                vmin=vmin_input.value, 
                vmax=vmax_input.value,
                layer_name="EnMAP Custom Composite"
            )
            
            # Display it!
            display(m)
            
        except Exception as e:
            print(f"Oops! Something went wrong: {e}")

# 4. Connect the button to the logic and display the dashboard
render_button.on_click(on_button_clicked)

# Organize the layout neatly
ui = widgets.VBox([
    file_input,
    widgets.HBox([r_band, g_band, b_band]),
    widgets.HBox([vmin_input, vmax_input]),
    render_button,
    output_map
])

# Show the dashboard
display(ui)

In [ ]:
import leafmap
import ipywidgets as widgets
from IPython.display import display
import rasterio
from rasterio.warp import transform_bounds
import os

# 1. Create the User Interface (UI) Elements
file_input = widgets.Textarea(
    value=r"C:\Users\shash\Downloads\projects\hstool\hybiomass_south_asia\hybiomass_dataset\South_Asia\01931\024.tif",
    placeholder='Enter full paths to .tif files (one per line)',
    description='File Paths:',
    layout=widgets.Layout(width='80%', height='100px')
)

# Band selection widgets
r_band = widgets.IntText(value=45, description='Red:', layout=widgets.Layout(width='150px'))
g_band = widgets.IntText(value=29, description='Green:', layout=widgets.Layout(width='150px'))
b_band = widgets.IntText(value=15, description='Blue:', layout=widgets.Layout(width='150px'))

# Contrast stretching widgets
vmin_input = widgets.IntText(value=110, description='Vmin:', layout=widgets.Layout(width='200px'))
vmax_input = widgets.IntText(value=2000, description='Vmax:', layout=widgets.Layout(width='200px'))

# The action button and output area
render_button = widgets.Button(description="Render Multi-Raster Map", button_style='primary', layout=widgets.Layout(width='300px'))
output_map = widgets.Output()

# 2. Helper function to calculate combined Lat/Lon center for multiple files
def get_combined_map_center(filepaths):
    min_left, min_bottom, max_right, max_top = float('inf'), float('inf'), float('-inf'), float('-inf')
    
    valid_paths = []
    for filepath in filepaths:
        filepath = filepath.strip()
        if os.path.exists(filepath):
            with rasterio.open(filepath) as src:
                bounds = src.bounds
                crs = src.crs
                # Transform the bounding box to EPSG:4326 (Lat/Lon)
                left, bottom, right, top = transform_bounds(crs, 'EPSG:4326', *bounds)
                
                # Update combined bounding box
                min_left = min(min_left, left)
                min_bottom = min(min_bottom, bottom)
                max_right = max(max_right, right)
                max_top = max(max_top, top)
                valid_paths.append(filepath)
    
    if not valid_paths:
        return None, None, []
        
    lat = (min_bottom + max_top) / 2
    lon = (min_left + max_right) / 2
    return lat, lon, valid_paths

# 3. The main logic
def on_button_clicked(b):
    with output_map:
        output_map.clear_output() 
        
        # Split textarea into list of paths
        raw_paths = file_input.value.split('\n')
        
        try:
            print("Calculating bounds and loading rasters...")
            lat, lon, valid_paths = get_combined_map_center(raw_paths)
            
            if lat is None:
                print("Error: No valid file paths found.")
                return

            # Initialize map
            m = leafmap.Map(center=[lat, lon], zoom=11)
            
            # Loop through each valid file and add it to the map
            for filepath in valid_paths:
                filename = os.path.basename(filepath)
                m.add_raster(
                    filepath, 
                    bands=[r_band.value, g_band.value, b_band.value], 
                    vmin=vmin_input.value, 
                    vmax=vmax_input.value,
                    layer_name=f"EnMAP - {filename}"
                )
            
            display(m)
            
        except Exception as e:
            print(f"Oops! Something went wrong: {e}")

# 4. Connect and Display
render_button.on_click(on_button_clicked)

# Layout setup
ui = widgets.VBox([
    widgets.HTML("<h2>EnMAP Multi-Raster Viewer</h2>"),
    file_input,
    widgets.Label("Band Selection:"),
    widgets.HBox([r_band, g_band, b_band]),
    widgets.Label("Contrast Stretching:"),
    widgets.HBox([vmin_input, vmax_input]),
    render_button,
    output_map
])

display(ui)

In [ ]:
import leafmap
import ipywidgets as widgets
from IPython.display import display, clear_output
import rasterio
from rasterio.warp import transform_bounds, transform
import os
import pandas as pd
import matplotlib.pyplot as plt

# Enable inline plotting for matplotlib
%matplotlib inline

# ==========================================
# 1. UI WIDGETS
# ==========================================

# File Inputs
file_input = widgets.Textarea(
    value=r"C:\Users\shash\Downloads\projects\hstool\hybiomass_south_asia\hybiomass_dataset\South_Asia\01931\024.tif",
    placeholder='Enter full paths to .tif files (one per line)',
    description='File Paths:',
    layout=widgets.Layout(width='80%', height='100px')
)

# Visualization Type Dropdown
vis_mode = widgets.Dropdown(
    options=["RGB Composite (Hyperspectral)", "Aboveground Biomass (AGBD)", "Plant Functional Type (PFT)"],
    value="RGB Composite (Hyperspectral)",
    description='Vis Type:',
    layout=widgets.Layout(width='400px')
)

# --- RGB Widgets ---
r_band = widgets.IntText(value=45, description='Red:', layout=widgets.Layout(width='120px'))
g_band = widgets.IntText(value=29, description='Green:', layout=widgets.Layout(width='120px'))
b_band = widgets.IntText(value=15, description='Blue:', layout=widgets.Layout(width='120px'))
rgb_vmin = widgets.IntText(value=110, description='Vmin:', layout=widgets.Layout(width='150px'))
rgb_vmax = widgets.IntText(value=2000, description='Vmax:', layout=widgets.Layout(width='150px'))
rgb_box = widgets.HBox([r_band, g_band, b_band, rgb_vmin, rgb_vmax])

# --- AGBD Widgets ---
agbd_cmap = widgets.Dropdown(options=['viridis', 'Greens', 'YlGn'], value='viridis', description='Colormap:', layout=widgets.Layout(width='200px'))
agbd_vmin = widgets.FloatText(value=0.0, description='Vmin:', layout=widgets.Layout(width='150px'))
agbd_vmax = widgets.FloatText(value=300.0, description='Vmax:', layout=widgets.Layout(width='150px'))
agbd_box = widgets.HBox([agbd_cmap, agbd_vmin, agbd_vmax])

# --- PFT Widgets ---
pft_cmap = widgets.Dropdown(options=['tab20', 'Set1', 'Paired'], value='tab20', description='Colormap:', layout=widgets.Layout(width='200px'))
pft_vmin = widgets.IntText(value=0, description='Vmin:', layout=widgets.Layout(width='150px'))
pft_vmax = widgets.IntText(value=15, description='Vmax:', layout=widgets.Layout(width='150px'))
pft_box = widgets.HBox([pft_cmap, pft_vmin, pft_vmax])

# Container for dynamic settings
settings_container = widgets.VBox([rgb_box])

# Action Buttons and Manual Inputs
render_button = widgets.Button(description="Render Map", button_style='primary')
man_lat = widgets.FloatText(value=0.0, description='Lat:')
man_lon = widgets.FloatText(value=0.0, description='Lon:')
extract_button = widgets.Button(description="Extract from Coords", button_style='info')
manual_input_box = widgets.HBox([man_lat, man_lon, extract_button])

# Output Targets
map_output = widgets.Output()
plot_output = widgets.Output()

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def update_ui(*args):
    """Swaps the parameter widgets based on the selected visualization mode."""
    if vis_mode.value == "RGB Composite (Hyperspectral)":
        settings_container.children = [rgb_box]
    elif vis_mode.value == "Aboveground Biomass (AGBD)":
        settings_container.children = [agbd_box]
    elif vis_mode.value == "Plant Functional Type (PFT)":
        settings_container.children = [pft_box]

vis_mode.observe(update_ui, 'value')

def get_combined_map_center(filepaths):
    min_left, min_bottom, max_right, max_top = float('inf'), float('inf'), float('-inf'), float('-inf')
    valid_paths = []
    for filepath in filepaths:
        filepath = filepath.strip()
        if os.path.exists(filepath):
            with rasterio.open(filepath) as src:
                left, bottom, right, top = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
                min_left, min_bottom = min(min_left, left), min(min_bottom, bottom)
                max_right, max_top = max(max_right, right), max(max_top, top)
                valid_paths.append(filepath)
    
    if not valid_paths:
        return None, None, []
    return (min_bottom + max_top) / 2, (min_left + max_right) / 2, valid_paths

def get_spectral_signature(filepaths, lat, lon):
    for filepath in filepaths:
        filepath = filepath.strip()
        if not os.path.exists(filepath): continue
        with rasterio.open(filepath) as src:
            try:
                xs, ys, *_ = transform('EPSG:4326', src.crs, [lon], [lat])
                x, y = xs[0], ys[0]
                if (src.bounds.left <= x <= src.bounds.right) and (src.bounds.bottom <= y <= src.bounds.top):
                    gen = src.sample([(x, y)], indexes=list(range(1, 203)))
                    values = list(gen)[0] 
                    return values, os.path.basename(filepath)
            except Exception:
                continue 
    return None, None

def plot_signature(lat, lon):
    """Extracts data and renders the matplotlib plot in the output widget."""
    raw_paths = [p for p in file_input.value.split('\n') if p.strip()]
    
    with plot_output:
        clear_output(wait=True)
        print(f"Extracting signature for Lat: {lat:.5f}, Lon: {lon:.5f}...")
        
        signature, source_file = get_spectral_signature(raw_paths, lat, lon)
        
        if signature is not None:
            clear_output(wait=True)
            print(f"Source: {source_file} | Lat: {lat:.5f} | Lon: {lon:.5f}")
            
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(range(1, 203), signature, color='blue', linewidth=1.5)
            ax.set_title("Spectral Signature")
            ax.set_xlabel("Band Index (1-202)")
            ax.set_ylabel("Reflectance")
            ax.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.show()
        else:
            clear_output(wait=True)
            print("🚨 Out of bounds! Coordinates do not fall within the spatial extent of any loaded .tif files.")

# ==========================================
# 3. EVENT HANDLERS
# ==========================================

def on_map_interaction(**kwargs):
    """Handles map clicks to extract spectral data."""
    if kwargs.get('type') == 'click':
        coords = kwargs.get('coordinates')
        lat, lon = coords[0], coords[1]
        man_lat.value = lat
        man_lon.value = lon
        plot_signature(lat, lon)

def on_render_clicked(b):
    """Renders the map and adds the selected rasters."""
    with map_output:
        clear_output(wait=True)
        raw_paths = [p for p in file_input.value.split('\n') if p.strip()]
        
        lat, lon, valid_paths = get_combined_map_center(raw_paths)
        if lat is None:
            print("Error: No valid file paths found.")
            return

        print("Rendering map...")
        m = leafmap.Map(center=[lat, lon], zoom=12)
        
        # Determine current visualization parameters
        mode = vis_mode.value
        if mode == "RGB Composite (Hyperspectral)":
            bands = [r_band.value, g_band.value, b_band.value]
            cmap, vmin, vmax = None, rgb_vmin.value, rgb_vmax.value
        elif mode == "Aboveground Biomass (AGBD)":
            bands = [203]
            cmap, vmin, vmax = agbd_cmap.value, agbd_vmin.value, agbd_vmax.value
        elif mode == "Plant Functional Type (PFT)":
            bands = [204]
            cmap, vmin, vmax = pft_cmap.value, pft_vmin.value, pft_vmax.value
            
        for filepath in valid_paths:
            m.add_raster(
                filepath, bands=bands, cmap=cmap, 
                vmin=vmin, vmax=vmax, 
                layer_name=os.path.basename(filepath)
            )
            
        # Attach click event listener to the map
        m.on_interaction(on_map_interaction)
        
        clear_output(wait=True)
        display(m)

def on_extract_clicked(b):
    """Handles manual coordinate extraction."""
    plot_signature(man_lat.value, man_lon.value)

render_button.on_click(on_render_clicked)
extract_button.on_click(on_extract_clicked)

# ==========================================
# 4. DISPLAY LAYOUT
# ==========================================

ui_layout = widgets.VBox([
    widgets.HTML("<h2>EnMAP Interactive Composite Viewer</h2>"),
    file_input,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Visualization Settings</b>"),
    vis_mode,
    settings_container,
    render_button,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Map View (Click to extract spectral signature)</b>"),
    map_output,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Spectral Analysis</b>"),
    manual_input_box,
    plot_output
])

display(ui_layout)